In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper: convert timestamp series (seconds) to Asia/Shanghai tz-aware pandas Timestamps
tz = pytz.timezone('Asia/Shanghai')
def to_shanghai_ts(series):
    return pd.to_datetime(series, unit='s', utc=True).dt.tz_convert(tz)

# Summarizer for metric/trace/log files
def summarize_kv(df, name_col):
    total_rows = int(df.shape[0])
    ts_local = to_shanghai_ts(df['timestamp'])
    min_ts = ts_local.min()
    max_ts = ts_local.max()
    # cmdb_id summary (top up to 20 by frequency)
    cmdb_counts = df['cmdb_id'].value_counts()
    distinct_cmdb_count = int(cmdb_counts.shape[0])
    top_cmdb = cmdb_counts.head(20).index.tolist()
    # name_col summary (counts top 20)
    if name_col in df.columns:
        name_counts = df[name_col].value_counts().head(20).reset_index()
        name_counts.columns = [name_col, 'count']
    else:
        name_counts = pd.DataFrame(columns=[name_col, 'count'])
    return {
        'total_rows': total_rows,
        'min_timestamp_shanghai': None if pd.isna(min_ts) else min_ts.isoformat(),
        'max_timestamp_shanghai': None if pd.isna(max_ts) else max_ts.isoformat(),
        'distinct_cmdb_count': distinct_cmdb_count,
        'top_cmdb_up_to_20': top_cmdb,
        'distinct_{}_count_estimate'.format(name_col): int(df[name_col].nunique()) if name_col in df.columns else 0,
        'top_{}_counts_up_to_20'.format(name_col): name_counts
    }

metric_summary = summarize_kv(df_metric, 'kpi_name')
trace_summary = summarize_kv(df_trace, 'trace_name')
log_summary = summarize_kv(df_log, 'log_name')

# Summarize error_logs: total count, min/max ts, distinct cmdbs, and up to 10 sample rows (timestamp shanghai, cmdb_id, message[:120])
ts_local_error = to_shanghai_ts(df_error['timestamp'])
error_min_ts = ts_local_error.min()
error_max_ts = ts_local_error.max()
error_cmdb_counts = df_error['cmdb_id'].value_counts()
error_samples = df_error[['timestamp', 'cmdb_id', 'message']].copy()
error_samples['timestamp_shanghai'] = to_shanghai_ts(error_samples['timestamp'])
error_samples['message_short'] = error_samples['message'].astype(str).str.slice(0, 120)
error_samples_small = error_samples[['timestamp_shanghai', 'cmdb_id', 'message_short']].head(10).reset_index(drop=True)

error_summary = {
    'total_rows': int(df_error.shape[0]),
    'min_timestamp_shanghai': None if pd.isna(error_min_ts) else error_min_ts.isoformat(),
    'max_timestamp_shanghai': None if pd.isna(error_max_ts) else error_max_ts.isoformat(),
    'distinct_cmdb_count': int(error_cmdb_counts.shape[0]),
    'top_cmdb_up_to_20': error_cmdb_counts.head(20).index.tolist(),
    'samples_up_to_10_rows': error_samples_small
}

# Combine into final JSON-like dict (compact)
summary = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': error_summary
}

summary

```
Out[1]:
```
```python
# Build a concise plain-English summary string from the previously computed `summary` dict.
s = summary  # reuse variable from prior execution

m = s['metric.csv']
t = s['trace.csv']
l = s['log.csv']
e = s['error_logs.csv']

# Helper to format top-name counts (DataFrame) into a short inline list (up to n items)
def fmt_top_df(df_like, n=6):
    try:
        df = df_like.head(n)
        return "; ".join(f"{row[0]} ({int(row[1])})" for row in df.values)
    except Exception:
        return ""

metric_top = fmt_top_df(m['top_kpi_name_counts_up_to_20'], 6)
trace_top = fmt_top_df(t['top_trace_name_counts_up_to_20'], 6)
log_top = fmt_top_df(l['top_log_name_counts_up_to_20'], 6)

summary_text = (
    "Telemetry summary (plain English):\n\n"
    "metric.csv:\n"
    f"- Rows: {m['total_rows']}\n"
    f"- Time range (Asia/Shanghai): {m['min_timestamp_shanghai']} to {m['max_timestamp_shanghai']}\n"
    f"- Distinct cmdb_id: {m['distinct_cmdb_count']}. Top cmdb_id (up to 20 shown): {', '.join(m['top_cmdb_up_to_20'])}\n"
    f"- Distinct kpi_name: {m['distinct_kpi_name_count_estimate']}. Top kpi_name counts (examples, up to 6): {metric_top}\n\n"
    "trace.csv:\n"
    f"- Rows: {t['total_rows']}\n"
    f"- Time range (Asia/Shanghai): {t['min_timestamp_shanghai']} to {t['max_timestamp_shanghai']}\n"
    f"- Distinct cmdb_id: {t['distinct_cmdb_count']}. Top cmdb_id (up to 20 shown): {', '.join(t['top_cmdb_up_to_20'])}\n"
    f"- Distinct trace_name: {t['distinct_trace_name_count_estimate']}. Top trace_name counts (examples, up to 6): {trace_top}\n\n"
    "log.csv:\n"
    f"- Rows: {l['total_rows']}\n"
    f"- Time range (Asia/Shanghai): {l['min_timestamp_shanghai']} to {l['max_timestamp_shanghai']}\n"
    f"- Distinct cmdb_id: {l['distinct_cmdb_count']}. Top cmdb_id (up to 20 shown): {', '.join(l['top_cmdb_up_to_20'])}\n"
    f"- Distinct log_name: {l['distinct_log_name_count_estimate']}. Top log_name counts: {log_top}\n\n"
    "error_logs.csv:\n"
    f"- Rows: {e['total_rows']}. No sample rows available.\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 325620, 'min_timestamp_shanghai': '2022-03-21T16:00:00+08:00', 'max_timestamp_shanghai': '2022-03-21T17:00:00+08:00', 'distinct_cmdb_count': 59, 'top_cmdb_up_to_20': ['adservice', 'adservice2', 'frontend2-0', 'frontend-2', 'frontend-1', 'frontend-0', 'checkoutservice-1', 'checkoutservice2-0', 'checkoutservice-2', 'checkoutservice-0', 'productcatalogservice-1', 'productcatalogservice-0', 'productcatalogservice-2', 'productcatalogservice2-0', 'adservice2-0', 'recommendationservice2-0', 'cartservice2-0', 'cartservice-2', 'cartservice-0', 'recommendationservice-2'], 'distinct_kpi_name_count_estimate': 1428, 'top_kpi_name_counts_up_to_20':                                              kpi_name  count
0   container.node-6.container_network_transmit_pa...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_transmit_er...   2196
3   container.node-6.container_network_receive_pac...   2196
4   container.node-6.container_network_transmit_MB...   2196
5   container.node-6.container_network_receive_pac...   2196
6   container.node-6.container_network_receive_err...   2196
7   container.node-6.container_network_receive_MB....   2196
8   container.node-6.container_spec_memory_swap_li...   2074
9   container.node-6.container_fs_io_time_seconds....   2074
10  container.node-6.container_fs_writes_merged./d...   2074
11            container.node-6.container_memory_cache   2074
12  container.node-6.container_memory_failures.hie...   2074
13     container.node-6.container_fs_inodes./dev/vda1   2074
14   container.node-6.container_fs_usage_MB./dev/vda1   2074
15  container.node-6.container_fs_write_seconds./d...   2074
16         container.node-6.container_spec_cpu_period   2074
17                 container.node-6.container_sockets   2074
18          container.node-6.container_spec_cpu_quota   2074
19  container.node-6.container_fs_io_current./dev/...   2074}, 'trace.csv': {'total_rows': 36304, 'min_timestamp_shanghai': '2022-03-21T16:30:00+08:00', 'max_timestamp_shanghai': '2022-03-21T16:59:00+08:00', 'distinct_cmdb_count': 40, 'top_cmdb_up_to_20': ['frontend-1', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice-0', 'frontend-2', 'frontend-0', 'productcatalogservice-2', 'productcatalogservice-1', 'productcatalogservice-0', 'frontend2-0', 'checkoutservice2-0', 'recommendationservice-0', 'recommendationservice-2', 'cartservice-0', 'cartservice-2', 'cartservice-1', 'recommendationservice-1', 'shippingservice-1', 'shippingservice-2', 'currencyservice-2'], 'distinct_trace_name_count_estimate': 200, 'top_trace_name_counts_up_to_20':                                     trace_name  count
0              trace.from_frontend-1.row_count    630
1             trace.from_frontend-1.error_rate    630
2           trace.from_frontend-1.duration_p95    630
3          trace.from_frontend-1.duration_mean    630
4              trace.from_frontend-2.row_count    600
5             trace.from_frontend-2.error_rate    600
6           trace.from_frontend-2.duration_p95    600
7          trace.from_frontend-2.duration_mean    600
8          trace.from_frontend-0.duration_mean    598
9           trace.from_frontend-0.duration_p95    598
10            trace.from_frontend-0.error_rate    598
11             trace.from_frontend-0.row_count    598
12  trace.from_checkoutservice-1.duration_mean    536
13   trace.from_checkoutservice-1.duration_p95    536
14     trace.from_checkoutservice-1.error_rate    536
15      trace.from_checkoutservice-1.row_count    536
16      trace.from_checkoutservice-2.row_count    533
17     trace.from_checkoutservice-2.error_rate    533
18   trace.from_checkoutservice-2.duration_p95    533
19  trace.from_checkoutservice-2.duration_mean    533}, 'log.csv': {'total_rows': 1832, 'min_timestamp_shanghai': '2022-03-21T16:30:00+08:00', 'max_timestamp_shanghai': '2022-03-21T17:00:00+08:00', 'distinct_cmdb_count': 31, 'top_cmdb_up_to_20': ['cartservice-0', 'cartservice-2', 'currencyservice-1', 'currencyservice-0', 'currencyservice-2', 'frontend-1', 'frontend-0', 'recommendationservice-0', 'shippingservice-2', 'adservice-1', 'checkoutservice-2', 'checkoutservice-0', 'cartservice-1', 'adservice-2', 'adservice-0', 'emailservice-2', 'emailservice-1', 'emailservice-0', 'checkoutservice-1', 'paymentservice-1'], 'distinct_log_name_count_estimate': 2, 'top_log_name_counts_up_to_20':           log_name  count
0  log.error_count    916
1    log.row_count    916}, 'error_logs.csv': {'total_rows': 0, 'min_timestamp_shanghai': None, 'max_timestamp_shanghai': None, 'distinct_cmdb_count': 0, 'top_cmdb_up_to_20': [], 'samples_up_to_10_rows': Empty DataFrame
Columns: [timestamp_shanghai, cmdb_id, message_short]
Index: []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import numpy as np

# Reuse df_metric if exists, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timestamp as tz-aware Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
df_metric['timestamp_shanghai'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 and counts per (cmdb_id, kpi_name) using full series
grouped = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
agg_counts = grouped.count().rename('count')
agg_p95 = grouped.quantile(0.95).rename('p95')
agg_p5 = grouped.quantile(0.05).rename('p5')

thresholds = pd.concat([agg_counts, agg_p95, agg_p5], axis=1).reset_index()
thresholds['low_data_flag'] = thresholds['count'] < 10

# 2) Filter to incident window inclusive: 2022-03-21 16:30:00+08:00 through 2022-03-21 17:00:00+08:00
start = tz.localize(pd.Timestamp('2022-03-21 16:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 17:00:00'))
mask_window = (df_metric['timestamp_shanghai'] >= start) & (df_metric['timestamp_shanghai'] <= end)
df_window = df_metric.loc[mask_window].copy()

# Merge thresholds into window rows
df_win = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('', '_thr'))

# 3) Mark anomalies and compute severities
# handle possible NaNs in p95/p5 by not marking anomalies where thresholds missing
df_win['anomaly_high'] = df_win['value'] >= df_win['p95']
df_win['anomaly_low'] = df_win['value'] <= df_win['p5']
df_win['anomaly'] = df_win['anomaly_high'] | df_win['anomaly_low']

# severity calculations with small epsilon
eps = 1e-9
df_win['severity_high'] = np.where(df_win['anomaly_high'], (df_win['value'] - df_win['p95']) / (df_win['p95'] + eps), 0.0)
df_win['severity_low'] = np.where(df_win['anomaly_low'], (df_win['p5'] - df_win['value']) / (df_win['p5'] + eps), 0.0)
df_win['severity'] = df_win[['severity_high', 'severity_low']].max(axis=1)

# Keep only anomaly rows for aggregation
anoms = df_win[df_win['anomaly']].copy()

# 4) Aggregate anomalies per cmdb_id
if not anoms.empty:
    # per cmdb_id,kpi_name stats within window
    per_kpi = (anoms.groupby(['cmdb_id', 'kpi_name'])
                  .agg(anomaly_count=('anomaly', 'size'),
                       earliest_anom_ts=('timestamp_shanghai', 'min'),
                       max_severity=('severity', 'max'))
                  .reset_index())
    # attach low_data_flag from thresholds
    per_kpi = per_kpi.merge(thresholds[['cmdb_id','kpi_name','low_data_flag']], on=['cmdb_id','kpi_name'], how='left')

    # per cmdb_id aggregation
    per_node = (per_kpi.groupby('cmdb_id')
                   .agg(anomalous_point_count=('anomaly_count', 'sum'),
                        distinct_anomalous_kpi_count=('kpi_name', 'nunique'),
                        earliest_anomaly_timestamp=('earliest_anom_ts', 'min'),
                        max_severity=('max_severity', 'max'))
                   .reset_index())

    # build top up to 5 kpis per cmdb_id sorted by anomaly_count desc
    def top_kpis_for_node(df_kpis):
        df_sorted = df_kpis.sort_values(['anomaly_count','max_severity'], ascending=[False, False]).head(5)
        # make compact tuples: (kpi_name, anomaly_count, earliest_ts_iso, max_severity, low_data_flag)
        return list(df_sorted.apply(lambda r: (r['kpi_name'],
                                              int(r['anomaly_count']),
                                              pd.Timestamp(r['earliest_anom_ts']).tz_convert(tz).isoformat()
                                              if not pd.isna(r['earliest_anom_ts']) else None,
                                              float(r['max_severity']),
                                              bool(r.get('low_data_flag', False))
                                              ), axis=1))

    top_kpis_series = per_kpi.groupby('cmdb_id').apply(top_kpis_for_node).rename('top_kpis_list').reset_index()
    per_node = per_node.merge(top_kpis_series, on='cmdb_id', how='left')

    # format earliest_anomaly_timestamp as iso strings
    per_node['earliest_anomaly_timestamp_shanghai'] = per_node['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    per_node = per_node.drop(columns=['earliest_anomaly_timestamp'])
    # sort by anomalous_point_count desc
    per_node_sorted = per_node.sort_values('anomalous_point_count', ascending=False).reset_index(drop=True)
else:
    per_kpi = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anom_ts','max_severity','low_data_flag'])
    per_node_sorted = pd.DataFrame(columns=['cmdb_id','anomalous_point_count','distinct_anomalous_kpi_count','max_severity','top_kpis_list','earliest_anomaly_timestamp_shanghai'])

# 5) Build compact result dict
total_anomalous_points_in_window = int(anoms.shape[0])
number_of_distinct_cmdb_with_anomalies = int(per_node_sorted.shape[0])

# top 20 cmdbs by anomaly count (limited)
top_20 = per_node_sorted.head(20).copy()
# Keep columns in desired order and compact types
top_20 = top_20[['cmdb_id','anomalous_point_count','distinct_anomalous_kpi_count','earliest_anomaly_timestamp_shanghai','max_severity','top_kpis_list']]

# 6) node-level cmdb_ids in top 20
node_rows = top_20[top_20['cmdb_id'].str.startswith('node-')]
node_level_list = node_rows[['cmdb_id','anomalous_point_count']].reset_index(drop=True)

# Also flag low_data per top kpi entries (already included in tuple as last element)

result = {
    'total_anomalous_points_in_window': total_anomalous_points_in_window,
    'number_of_distinct_cmdb_with_anomalies': number_of_distinct_cmdb_with_anomalies,
    'top_20_cmdbs_by_anomaly_count': top_20,
    'node_level_cmdbs_in_top_20': node_level_list
}

result

```
Out[2]:
```
```python
# Build a concise plain-English summary from the existing `result` variable
res = result  # reuse variable from prior execution

top_df = res['top_20_cmdbs_by_anomaly_count']
n_total = int(res['total_anomalous_points_in_window'])
n_cmdb = int(res['number_of_distinct_cmdb_with_anomalies'])

lines = []
lines.append(f"Metric-level anomaly detection summary:")
lines.append(f"- Total anomalous points in window: {n_total}")
lines.append(f"- Number of distinct cmdb_id with anomalies: {n_cmdb}")
lines.append(f"- Top cmdbs by anomalous point count (top 10 shown):")

# Format top rows (up to 10) with key fields
for i, row in top_df.head(10).iterrows():
    cm = row['cmdb_id']
    cnt = int(row['anomalous_point_count'])
    dk = int(row['distinct_anomalous_kpi_count'])
    ts = row['earliest_anomaly_timestamp_shanghai']
    sev = float(row['max_severity']) if not pd.isna(row['max_severity']) else 0.0
    sev_s = f"{sev:.4g}"
    # summarize up to 2 top kpis for brevity (include low_data_flag if present)
    kpis = row['top_kpis_list'] if isinstance(row['top_kpis_list'], list) else []
    kpi_summary = []
    for kp in kpis[:2]:
        # kp tuple: (kpi_name, anomaly_count, earliest_ts, max_severity, low_data_flag)
        name = kp[0]
        kcnt = int(kp[1])
        ldf = kp[4] if len(kp) >= 5 else False
        kpi_summary.append(f"{name} ({kcnt}{', low_data' if ldf else ''})")
    kpi_str = "; ".join(kpi_summary) if kpi_summary else "[]"
    lines.append(f"  {i+1}. {cm}: {cnt} anomalies, {dk} distinct KPIs, earliest {ts}, max_sev={sev_s}. Top KPIs: {kpi_str}")

# Node-level cmdbs in top 20
nodes_df = res['node_level_cmdbs_in_top_20']
if hasattr(nodes_df, 'empty') and nodes_df.empty:
    lines.append("- No node-* entries appear in the top 20 cmdb list.")
else:
    node_list = ", ".join(f"{r['cmdb_id']}({int(r['anomalous_point_count'])})" for _, r in nodes_df.iterrows())
    lines.append(f"- Node-level cmdbs in top 20: {node_list}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_anomalous_points_in_window': 122570, 'number_of_distinct_cmdb_with_anomalies': 59, 'top_20_cmdbs_by_anomaly_count':                      cmdb_id  anomalous_point_count  distinct_anomalous_kpi_count earliest_anomaly_timestamp_shanghai  max_severity                                      top_kpis_list
0                  adservice                   9547                           341            2022-03-21T16:30:00+0800  4.689846e-01  [(app.http.count, 31, 2022-03-21T16:30:00+08:0...
1                 adservice2                   9045                           333            2022-03-21T16:30:00+0800  1.000000e+09  [(runtime.java_lang_Memory_ObjectPendingFinali...
2                frontend2-0                   4677                           208            2022-03-21T16:30:00+0800  1.000000e+00  [(container.node-6.container_cpu_load_average_...
3                 frontend-0                   4272                           192            2022-03-21T16:30:00+0800  9.067263e-01  [(container.node-6.container_cpu_cfs_throttled...
4                 frontend-1                   4098                           191            2022-03-21T16:30:00+0800  2.200000e+10  [(container.node-5.container_memory_failures.c...
5                 frontend-2                   4040                           189            2022-03-21T16:30:00+0800  7.439021e-01  [(container.node-5.container_cpu_cfs_throttled...
6          checkoutservice-0                   3080                           143            2022-03-21T16:30:00+0800  1.500000e+09  [(container.node-6.container_fs_reads./dev/vda...
7         checkoutservice2-0                   3077                           148            2022-03-21T16:30:00+0800  2.045454e+00  [(container.node-6.container_sockets, 31, 2022...
8          checkoutservice-1                   3027                           148            2022-03-21T16:30:00+0800  2.342644e+00  [(container.node-6.container_cpu_load_average_...
9          checkoutservice-2                   2722                           144            2022-03-21T16:30:00+0800  2.237607e+01  [(container.node-5.container_cpu_load_average_...
10   productcatalogservice-0                   2677                           120            2022-03-21T16:30:00+0800  2.404760e+00  [(container.node-6.container_cpu_cfs_throttled...
11   productcatalogservice-1                   2604                           120            2022-03-21T16:30:00+0800  4.357139e+00  [(container.node-6.container_cpu_cfs_throttled...
12  recommendationservice2-0                   2549                           102            2022-03-21T16:30:00+0800  1.521095e+00  [(mesh.source.recommendationservice2.jaeger-co...
13   productcatalogservice-2                   2496                           115            2022-03-21T16:30:00+0800  8.467739e-01  [(container.node-6.container_cpu_cfs_throttled...
14              adservice2-0                   2491                           102            2022-03-21T16:30:00+0800  3.410146e+00  [(container.node-6.container_cpu_cfs_throttled...
15   recommendationservice-0                   2412                            98            2022-03-21T16:30:00+0800  2.857141e+00  [(container.node-6.container_cpu_cfs_throttled...
16   recommendationservice-2                   2411                            98            2022-03-21T16:30:00+0800  2.499999e-01  [(container.node-6.container_cpu_cfs_throttled...
17   recommendationservice-1                   2400                            98            2022-03-21T16:30:00+0800  1.452380e+00  [(container.node-6.container_cpu_cfs_throttled...
18  productcatalogservice2-0                   2364                           102            2022-03-21T16:30:00+0800  6.923561e-01  [(container.node-6.container_cpu_load_average_...
19             cartservice-2                   2321                           100            2022-03-21T16:30:00+0800  3.109127e+00  [(container.node-6.container_cpu_cfs_throttled..., 'node_level_cmdbs_in_top_20': Empty DataFrame
Columns: [cmdb_id, anomalous_point_count]
Index: []}```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import re
import numpy as np

# Reuse df_trace if present, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timestamp as tz-aware Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
df_trace['timestamp_shanghai'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per trace_name and counts; flag low data (<10)
grp = df_trace.groupby('trace_name')['value']
thr_counts = grp.count().rename('count')
thr_p95 = grp.quantile(0.95).rename('p95')
thr_p5 = grp.quantile(0.05).rename('p5')
thr = pd.concat([thr_counts, thr_p95, thr_p5], axis=1).reset_index()
thr['low_data_flag'] = thr['count'] < 10

# Merge thresholds into df_trace
df_tr = df_trace.merge(thr, on='trace_name', how='left')

# 2) Mark anomalies according to rules
# conditions
is_error_rate = df_tr['trace_name'].str.contains('error_rate', na=False)
is_row_count = df_tr['trace_name'].str.contains('row_count', na=False)
is_duration = df_tr['trace_name'].str.contains('duration_mean|duration_p95', regex=True, na=False)

# apply anomaly logic
df_tr['anomaly_high'] = ((is_error_rate | is_duration) & (df_tr['value'] >= df_tr['p95']))
df_tr['anomaly_low'] = (is_row_count & (df_tr['value'] <= df_tr['p5']))
df_tr['anomaly'] = df_tr['anomaly_high'] | df_tr['anomaly_low']

# 3) Compute severity
eps = 1e-9
df_tr['severity_high'] = np.where(df_tr['anomaly_high'], (df_tr['value'] - df_tr['p95']) / (df_tr['p95'] + eps), 0.0)
df_tr['severity_low'] = np.where(df_tr['anomaly_low'], (df_tr['p5'] - df_tr['value']) / (df_tr['p5'] + eps), 0.0)
df_tr['severity'] = df_tr[['severity_high', 'severity_low']].max(axis=1)

# 4) Extract source component from trace_name: part after 'trace.from_' and before next dot
def extract_source(tn):
    if not isinstance(tn, str):
        return None
    m = re.match(r'^trace\.from_([^\.]+)\.', tn)
    return m.group(1) if m else None

# compute source for each row (will be repeated, but acceptable)
df_tr['source'] = df_tr['trace_name'].apply(extract_source)

# Focus on anomalies
anoms = df_tr[df_tr['anomaly']].copy()

# If no anomalies, prepare empty outputs
if anoms.empty:
    total_anomalous_trace_points = 0
    number_of_distinct_trace_sources_with_anomalies = 0
    top_sources_df = pd.DataFrame(columns=['source','anomalous_point_count','distinct_anomalous_trace_name_count','earliest_anomaly_timestamp_shanghai','max_severity','top_trace_names_list'])
    top_sources_list = pd.DataFrame(columns=['source','anomalous_point_count']).head(0)
else:
    # Per trace_name aggregation (within anomalies)
    per_trace = (anoms.groupby(['trace_name', 'source'])
                    .agg(anomaly_count=('anomaly', 'size'),
                         earliest_anom_ts=('timestamp_shanghai', 'min'),
                         max_severity=('severity', 'max'))
                    .reset_index())

    # attach low_data_flag from thr
    per_trace = per_trace.merge(thr[['trace_name','low_data_flag']], on='trace_name', how='left')

    # Format earliest_anom_ts to iso string
    per_trace['earliest_anom_ts_iso'] = per_trace['earliest_anom_ts'].dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

    # Per source aggregation
    per_source = (per_trace.groupby('source')
                      .agg(anomalous_point_count=('anomaly_count', 'sum'),
                           distinct_anomalous_trace_name_count=('trace_name', 'nunique'),
                           earliest_anomaly_timestamp=('earliest_anom_ts', 'min'),
                           max_severity=('max_severity', 'max'))
                      .reset_index())

    # Build top up to 5 trace_names per source
    def top_traces_for_source(df_traces):
        df_sorted = df_traces.sort_values(['anomaly_count','max_severity'], ascending=[False, False]).head(5)
        # return tuples: (trace_name, anomaly_count, earliest_ts_iso, max_severity, low_data_flag)
        return list(df_sorted.apply(lambda r: (r['trace_name'],
                                              int(r['anomaly_count']),
                                              r['earliest_anom_ts_iso'],
                                              float(r['max_severity']),
                                              bool(r.get('low_data_flag', False))
                                              ), axis=1))

    top_traces_series = per_trace.groupby('source').apply(top_traces_for_source).rename('top_trace_names_list').reset_index()
    per_source = per_source.merge(top_traces_series, on='source', how='left')

    # format earliest timestamp iso
    per_source['earliest_anomaly_timestamp_shanghai'] = per_source['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    per_source = per_source.drop(columns=['earliest_anomaly_timestamp'])

    # sort sources by anomalous_point_count desc
    per_source_sorted = per_source.sort_values('anomalous_point_count', ascending=False).reset_index(drop=True)

    # totals and compact lists
    total_anomalous_trace_points = int(anoms.shape[0])
    number_of_distinct_trace_sources_with_anomalies = int(per_source_sorted.shape[0])

    # top sources DataFrame limited to top 20 for compactness
    top_sources_df = per_source_sorted.head(20)[['source','anomalous_point_count','distinct_anomalous_trace_name_count','earliest_anomaly_timestamp_shanghai','max_severity','top_trace_names_list']]

    # compact list up to top 10 sources sorted by anomalous_point_count
    top_sources_list = top_sources_df[['source','anomalous_point_count']].head(10).reset_index(drop=True)

# Build result dict (compact DataFrames)
result_trace_anomalies = {
    'total_anomalous_trace_points': total_anomalous_trace_points,
    'number_of_distinct_trace_sources_with_anomalies': number_of_distinct_trace_sources_with_anomalies,
    'top_trace_sources_up_to_20': top_sources_df,
    'top_trace_sources_list_up_to_10': top_sources_list,
    # also include per-trace summary for top 20 sources' traces (compact): gather per_trace filtered to those sources
    'per_trace_anomaly_summary_for_top_sources': (per_trace[per_trace['source'].isin(top_sources_df['source'])][['source','trace_name','anomaly_count','earliest_anom_ts_iso','max_severity','low_data_flag']]
                                                 .sort_values(['source','anomaly_count'], ascending=[True, False])
                                                 .groupby('source').head(5).reset_index(drop=True)) if not anoms.empty else pd.DataFrame(columns=['source','trace_name','anomaly_count','earliest_anom_ts_iso','max_severity','low_data_flag'])
}

result_trace_anomalies

```
Out[3]:
```
```python
# Build a concise plain-English summary from the previously computed `result_trace_anomalies`
try:
    res = result_trace_anomalies
except NameError:
    # fallback: try variable name from prior cell
    res = locals().get('result_trace_anomalies', None)

if res is None:
    summary = "No trace anomaly result available in the current Python state."
else:
    top_df = res['top_trace_sources_up_to_20']
    total = int(res['total_anomalous_trace_points'])
    n_sources = int(res['number_of_distinct_trace_sources_with_anomalies'])

    lines = []
    lines.append("Trace-level anomaly detection summary (plain English):")
    lines.append(f"- Total anomalous trace points: {total}")
    lines.append(f"- Number of distinct trace sources with anomalies: {n_sources}")
    lines.append("- Top trace sources by anomalous point count (up to 10 shown):")

    # show up to 10 top sources with compact info
    for i, row in top_df.head(10).iterrows():
        src = row['source']
        cnt = int(row['anomalous_point_count'])
        distinct = int(row['distinct_anomalous_trace_name_count'])
        earliest = row['earliest_anomaly_timestamp_shanghai']
        max_sev = float(row['max_severity']) if not pd.isna(row['max_severity']) else 0.0
        # summarize up to 2 top trace names for brevity
        top_traces = row['top_trace_names_list'] if isinstance(row['top_trace_names_list'], list) else []
        trace_summaries = []
        for t in top_traces[:2]:
            # tuple: (trace_name, anomaly_count, earliest_ts_iso, max_severity, low_data_flag)
            name = t[0]; tcnt = int(t[1]); ldf = t[4] if len(t) >= 5 else False
            trace_summaries.append(f"{name} ({tcnt}{', low_data' if ldf else ''})")
        traces_str = "; ".join(trace_summaries) if trace_summaries else "[]"
        lines.append(f"  {i+1}. {src}: {cnt} anomalies, {distinct} distinct trace metrics, earliest {earliest}, max_sev={max_sev:.4g}. Top traces: {traces_str}")

    lines.append("- Note: low_data_flag marks trace_names with <10 total data points in their full series (shown per-trace above if present).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'total_anomalous_trace_points': 11721, 'number_of_distinct_trace_sources_with_anomalies': 13, 'top_trace_sources_up_to_20':                       source  anomalous_point_count  distinct_anomalous_trace_name_count earliest_anomaly_timestamp_shanghai  max_severity                               top_trace_names_list
0                 frontend-1                    758                                    4            2022-03-21T16:30:00+0800      1.412219  [(trace.from_frontend-1.error_rate, 630, 2022-...
1                 frontend-0                    723                                    4            2022-03-21T16:30:00+0800      2.675147  [(trace.from_frontend-0.error_rate, 598, 2022-...
2                 frontend-2                    720                                    4            2022-03-21T16:30:00+0800      1.846108  [(trace.from_frontend-2.error_rate, 600, 2022-...
3          checkoutservice-1                    700                                    4            2022-03-21T16:30:00+0800     11.720284  [(trace.from_checkoutservice-1.error_rate, 536...
4          checkoutservice-0                    699                                    4            2022-03-21T16:30:00+0800      0.796288  [(trace.from_checkoutservice-0.error_rate, 531...
5          checkoutservice-2                    694                                    4            2022-03-21T16:30:00+0800     15.179037  [(trace.from_checkoutservice-2.error_rate, 533...
6                       root                    284                                    4            2022-03-21T16:30:00+0800      2.999115  [(trace.from_root.error_rate, 240, 2022-03-21T...
7                frontend2-0                    247                                    4            2022-03-21T16:30:00+0800      0.489785  [(trace.from_frontend2-0.error_rate, 210, 2022...
8         checkoutservice2-0                    229                                    4            2022-03-21T16:30:00+0800      2.052211  [(trace.from_checkoutservice2-0.error_rate, 18...
9    recommendationservice-0                    120                                    4            2022-03-21T16:30:00+0800      0.512183  [(trace.from_recommendationservice-0.error_rat...
10   recommendationservice-1                    119                                    4            2022-03-21T16:30:00+0800      3.287197  [(trace.from_recommendationservice-1.error_rat...
11   recommendationservice-2                    119                                    4            2022-03-21T16:30:00+0800      0.160300  [(trace.from_recommendationservice-2.error_rat...
12  recommendationservice2-0                    107                                    4            2022-03-21T16:30:00+0800      0.251277  [(trace.from_recommendationservice2-0.error_ra..., 'top_trace_sources_list_up_to_10':                     source  anomalous_point_count
0               frontend-1                    758
1               frontend-0                    723
2               frontend-2                    720
3        checkoutservice-1                    700
4        checkoutservice-0                    699
5        checkoutservice-2                    694
6                     root                    284
7              frontend2-0                    247
8       checkoutservice2-0                    229
9  recommendationservice-0                    120, 'per_trace_anomaly_summary_for_top_sources':                       source                                        trace_name  anomaly_count      earliest_anom_ts_iso  max_severity  low_data_flag
0          checkoutservice-0           trace.from_checkoutservice-0.error_rate            531  2022-03-21T16:30:00+0800      0.000000          False
1          checkoutservice-0            trace.from_checkoutservice-0.row_count            113  2022-03-21T16:30:00+0800      0.000000          False
2          checkoutservice-0        trace.from_checkoutservice-0.duration_mean             28  2022-03-21T16:30:00+0800      0.301781          False
3          checkoutservice-0         trace.from_checkoutservice-0.duration_p95             27  2022-03-21T16:30:00+0800      0.796288          False
4          checkoutservice-1           trace.from_checkoutservice-1.error_rate            536  2022-03-21T16:30:00+0800      0.000000          False
..                       ...                                               ...            ...                       ...           ...            ...
47  recommendationservice2-0  trace.from_recommendationservice2-0.duration_p95              5  2022-03-21T16:31:00+0800      0.251277          False
48                      root                        trace.from_root.error_rate            240  2022-03-21T16:30:00+0800      0.000000          False
49                      root                         trace.from_root.row_count             20  2022-03-21T16:30:00+0800      0.375000          False
50                      root                     trace.from_root.duration_mean             12  2022-03-21T16:34:00+0800      0.284568          False
51                      root                      trace.from_root.duration_p95             12  2022-03-21T16:34:00+0800      2.999115          False

[52 rows x 6 columns]}

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse df_log if available, else load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure timestamp as tz-aware Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
df_log['timestamp_shanghai'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 16:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 17:00:00'))
mask_window = (df_log['timestamp_shanghai'] >= start) & (df_log['timestamp_shanghai'] <= end)
df_window = df_log.loc[mask_window].copy()

# Focus on log.error_count rows within window
df_err = df_window[df_window['log_name'] == 'log.error_count'].copy()

# Prepare empty-safe outputs
cols = ['cmdb_id','total_sum_error_count','minutes_with_errors','earliest_error_timestamp_shanghai','max_error_count']
if df_err.empty:
    summary_top20 = pd.DataFrame(columns=cols).head(0)
    raw_top3_logs = pd.DataFrame(columns=['timestamp_shanghai','cmdb_id','log_name','value']).head(0)
else:
    # Aggregate per cmdb_id
    grp = df_err.groupby('cmdb_id')
    total_sum = grp['value'].sum().rename('total_sum_error_count')
    minutes_with_errors = grp['value'].apply(lambda s: (s > 0).sum()).rename('minutes_with_errors')
    # earliest timestamp where value>0 (None if no positive)
    earliest = grp.apply(lambda g: g.loc[g['value'] > 0, 'timestamp_shanghai'].min()).rename('earliest_error_ts')
    max_err = grp['value'].max().rename('max_error_count')

    summary = pd.concat([total_sum, minutes_with_errors, earliest, max_err], axis=1).reset_index()
    # format earliest to ISO or None
    summary['earliest_error_timestamp_shanghai'] = summary['earliest_error_ts'].apply(
        lambda t: None if pd.isna(t) else pd.Timestamp(t).tz_convert(tz).isoformat())
    summary_top20 = summary[['cmdb_id','total_sum_error_count','minutes_with_errors','earliest_error_timestamp_shanghai','max_error_count']].sort_values(
        'total_sum_error_count', ascending=False).head(20).reset_index(drop=True)

    # For top 3 cmdb_id, return up to 10 raw log rows each (only log.error_count rows)
    top3 = summary_top20['cmdb_id'].head(3).tolist()
    if top3:
        df_top3_raw = df_err[df_err['cmdb_id'].isin(top3)].copy()
        df_top3_raw = df_top3_raw[['timestamp_shanghai','cmdb_id','log_name','value']].sort_values(['cmdb_id','timestamp_shanghai'])
        # keep up to 10 rows per cmdb_id
        raw_top3_logs = df_top3_raw.groupby('cmdb_id', sort=False).head(10).reset_index(drop=True)
    else:
        raw_top3_logs = pd.DataFrame(columns=['timestamp_shanghai','cmdb_id','log_name','value']).head(0)

# Result (compact)
result_log_analysis = {
    'summary_top20': summary_top20,
    'raw_top3_logs': raw_top3_logs
}

result_log_analysis

```
Out[4]:
```
```python
# Build a concise plain-English summary from the existing `result_log_analysis`
res = result_log_analysis  # reused from prior execution

summary_lines = []
summary_lines.append("Log analysis within 2022-03-21 16:30:00+08:00 to 2022-03-21 17:00:00+08:00:")

summary_lines.append("- Observation: All aggregated results for log.error_count are zero in the window.")
summary_lines.append("  * Top 20 cmdb_id (by total_sum_error_count) all have total_sum_error_count = 0.0, minutes_with_errors = 0, earliest_error_timestamp = None, max_error_count = 0.0.")
summary_lines.append("  * This indicates no non-zero log.error_count events were recorded in the incident window for the examined components.")

# Raw logs for top 3
raw = res.get('raw_top3_logs')
if raw is None or raw.empty:
    summary_lines.append("- Raw log samples for top-3 cmdb_id: none (no non-zero error_count rows).")
else:
    # report number of raw rows returned and that their values are zero
    n_raw_rows = raw.shape[0]
    # check if any value > 0
    any_positive = (raw['value'] > 0).any()
    summary_lines.append(f"- Raw log samples returned for top-3 cmdb_id: {n_raw_rows} rows (up to 10 per cmdb).")
    if any_positive:
        summary_lines.append("  * At least one sampled raw row has value > 0.")
    else:
        summary_lines.append("  * All sampled raw rows have log.error_count == 0.0 (no error minutes observed in samples).")

summary_lines.append("- Conclusion: No evidence of log.error_count anomalies during the incident window based on provided log.csv. If you want, I can show more raw rows or re-check other log metrics.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'summary_top20':               cmdb_id  total_sum_error_count  minutes_with_errors earliest_error_timestamp_shanghai  max_error_count
0         adservice-0                    0.0                    0                              None              0.0
1         adservice-1                    0.0                    0                              None              0.0
2         adservice-2                    0.0                    0                              None              0.0
3       cartservice-0                    0.0                    0                              None              0.0
4       cartservice-1                    0.0                    0                              None              0.0
5       cartservice-2                    0.0                    0                              None              0.0
6   checkoutservice-0                    0.0                    0                              None              0.0
7   checkoutservice-1                    0.0                    0                              None              0.0
8   checkoutservice-2                    0.0                    0                              None              0.0
9   currencyservice-0                    0.0                    0                              None              0.0
10  currencyservice-1                    0.0                    0                              None              0.0
11  currencyservice-2                    0.0                    0                              None              0.0
12     emailservice-0                    0.0                    0                              None              0.0
13     emailservice-1                    0.0                    0                              None              0.0
14     emailservice-2                    0.0                    0                              None              0.0
15         frontend-0                    0.0                    0                              None              0.0
16         frontend-1                    0.0                    0                              None              0.0
17         frontend-2                    0.0                    0                              None              0.0
18   paymentservice-0                    0.0                    0                              None              0.0
19   paymentservice-1                    0.0                    0                              None              0.0, 'raw_top3_logs':           timestamp_shanghai      cmdb_id         log_name  value
0  2022-03-21 16:30:00+08:00  adservice-0  log.error_count    0.0
1  2022-03-21 16:31:00+08:00  adservice-0  log.error_count    0.0
2  2022-03-21 16:32:00+08:00  adservice-0  log.error_count    0.0
3  2022-03-21 16:33:00+08:00  adservice-0  log.error_count    0.0
4  2022-03-21 16:34:00+08:00  adservice-0  log.error_count    0.0
..                       ...          ...              ...    ...
25 2022-03-21 16:35:00+08:00  adservice-2  log.error_count    0.0
26 2022-03-21 16:36:00+08:00  adservice-2  log.error_count    0.0
27 2022-03-21 16:37:00+08:00  adservice-2  log.error_count    0.0
28 2022-03-21 16:38:00+08:00  adservice-2  log.error_count    0.0
29 2022-03-21 16:39:00+08:00  adservice-2  log.error_count    0.0

[30 rows x 4 columns]}

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import re
import numpy as np

# Reuse df_metric if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timestamp_shanghai exists and is tz-aware Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if 'timestamp_shanghai' not in df_metric.columns:
    df_metric['timestamp_shanghai'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure dtype is tz-aware (convert if naive)
    df_metric['timestamp_shanghai'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95/P5 and counts per (cmdb_id, kpi_name) if not already computed
try:
    thresholds  # reuse from prior cells if present
except NameError:
    grp_full = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    agg_counts = grp_full.count().rename('count')
    agg_p95 = grp_full.quantile(0.95).rename('p95')
    agg_p5 = grp_full.quantile(0.05).rename('p5')
    thresholds = pd.concat([agg_counts, agg_p95, agg_p5], axis=1).reset_index()
    thresholds['low_data_flag'] = thresholds['count'] < 10

# 2) Extract node identifier from kpi_name: first substring matching 'node-\w+'
# We'll add a column 'node_id' to thresholds (for reference) and to window rows.
thresholds['node_id'] = thresholds['kpi_name'].str.extract(r'(node-\w+)', expand=False)

# 3) Filter metric.csv to the incident window inclusive
start = tz.localize(pd.Timestamp('2022-03-21 16:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 17:00:00'))
mask_window = (df_metric['timestamp_shanghai'] >= start) & (df_metric['timestamp_shanghai'] <= end)
df_win = df_metric.loc[mask_window].copy()

# Extract node_id on window rows from kpi_name
df_win['node_id'] = df_win['kpi_name'].str.extract(r'(node-\w+)', expand=False)

# Ignore rows without a node- substring for node aggregation step
df_win_nodes = df_win[df_win['node_id'].notna()].copy()

# If no node labels found, prepare none-safe empty outputs
if df_win_nodes.empty:
    node_summary_top10 = pd.DataFrame(columns=['node_id','anomalous_point_count','distinct_cmdb_ids_affected','earliest_anomaly_timestamp_shanghai','max_severity']).head(0)
    examples_top3 = pd.DataFrame(columns=['node_id','cmdb_id','kpi_name','anomaly_count','max_severity','low_data_flag']).head(0)
else:
    # 4) Merge thresholds (global P95/P5 and low_data_flag) into window rows by (cmdb_id,kpi_name)
    df_win_nodes = df_win_nodes.merge(thresholds[['cmdb_id','kpi_name','count','p95','p5','low_data_flag']], on=['cmdb_id','kpi_name'], how='left')

    # Mark anomalies (value >= p95 or value <= p5). Must handle possible NaNs in p95/p5.
    eps = 1e-9
    df_win_nodes['anomaly_high'] = df_win_nodes['p95'].notna() & (df_win_nodes['value'] >= df_win_nodes['p95'])
    df_win_nodes['anomaly_low'] = df_win_nodes['p5'].notna() & (df_win_nodes['value'] <= df_win_nodes['p5'])
    df_win_nodes['anomaly'] = df_win_nodes['anomaly_high'] | df_win_nodes['anomaly_low']

    # 4) Compute severity
    df_win_nodes['severity_high'] = np.where(df_win_nodes['anomaly_high'], (df_win_nodes['value'] - df_win_nodes['p95']) / (df_win_nodes['p95'] + eps), 0.0)
    df_win_nodes['severity_low'] = np.where(df_win_nodes['anomaly_low'], (df_win_nodes['p5'] - df_win_nodes['value']) / (df_win_nodes['p5'] + eps), 0.0)
    df_win_nodes['severity'] = df_win_nodes[['severity_high','severity_low']].max(axis=1)

    # Keep only anomalous rows
    anoms_nodes = df_win_nodes[df_win_nodes['anomaly']].copy()

    if anoms_nodes.empty:
        node_summary_top10 = pd.DataFrame(columns=['node_id','anomalous_point_count','distinct_cmdb_ids_affected','earliest_anomaly_timestamp_shanghai','max_severity']).head(0)
        examples_top3 = pd.DataFrame(columns=['node_id','cmdb_id','kpi_name','anomaly_count','max_severity','low_data_flag']).head(0)
    else:
        # 5) Aggregate anomalous rows by node_id
        per_node = (anoms_nodes.groupby('node_id')
                    .agg(anomalous_point_count=('anomaly', 'size'),
                         distinct_cmdb_ids_affected=('cmdb_id', 'nunique'),
                         earliest_anomaly_timestamp=('timestamp_shanghai', 'min'),
                         max_severity=('severity', 'max'))
                    .reset_index())

        per_node['earliest_anomaly_timestamp_shanghai'] = per_node['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
        per_node = per_node.drop(columns=['earliest_anomaly_timestamp'])

        node_summary_top10 = per_node.sort_values('anomalous_point_count', ascending=False).head(10).reset_index(drop=True)

        # 6) For each of the top 3 nodes, return up to 5 example KPI names (we report at (cmdb_id,kpi_name) granularity)
        top3_nodes = node_summary_top10['node_id'].head(3).tolist()

        # compute per (node_id, cmdb_id, kpi_name) aggregation
        per_pair = (anoms_nodes.groupby(['node_id','cmdb_id','kpi_name'])
                    .agg(anomaly_count=('anomaly', 'size'),
                         max_severity=('severity', 'max'))
                    .reset_index())

        # attach low_data_flag from thresholds: we have 'low_data_flag' in anoms_nodes; get unique low_data per pair
        lowdata = anoms_nodes.groupby(['node_id','cmdb_id','kpi_name'])['low_data_flag'].first().reset_index()
        per_pair = per_pair.merge(lowdata, on=['node_id','cmdb_id','kpi_name'], how='left')

        # For each node in top3, select up to 5 example KPI pairs sorted by anomaly_count desc then severity
        examples_list = []
        for nid in top3_nodes:
            df_n = per_pair[per_pair['node_id'] == nid].sort_values(['anomaly_count','max_severity'], ascending=[False, False]).head(5)
            # keep columns: node_id, cmdb_id, kpi_name, anomaly_count, max_severity, low_data_flag
            if not df_n.empty:
                df_n_sel = df_n[['node_id','cmdb_id','kpi_name','anomaly_count','max_severity','low_data_flag']].copy()
                examples_list.append(df_n_sel)
        if examples_list:
            examples_top3 = pd.concat(examples_list, ignore_index=True)
        else:
            examples_top3 = pd.DataFrame(columns=['node_id','cmdb_id','kpi_name','anomaly_count','max_severity','low_data_flag']).head(0)

# Final compact outputs: node_summary_top10 and examples_top3
node_summary_top10, examples_top3

```
Out[5]:
```
Summary of node-localized metric anomalies (incident window 2022-03-21 16:30–17:00 Asia/Shanghai):

- Top nodes by anomalous metric points:
  1. node-6
     - Anomalous points: 56,801
     - Distinct cmdb_id affected: 36
     - Earliest anomaly observed in window: 2022-03-21T16:30:00+08:00
     - Maximum observed severity: ~3.0e+09
  2. node-5
     - Anomalous points: 9,226
     - Distinct cmdb_id affected: 6
     - Earliest anomaly observed in window: 2022-03-21T16:30:00+08:00
     - Maximum observed severity: ~4.95e+10

- Example anomalous KPI names for the top nodes (up to 5 per node, with anomaly counts and max severity; low_data_flag indicates if the (cmdb_id,kpi_name) had <10 total points in full series):
  - node-6 (examples)
    - paymentservice-2 / container.node-6.container_fs_writes./dev/vda — count 31, max_severity ~3.0e+09, low_data_flag=False
    - paymentservice-2 / container.node-6.container_fs_reads./dev/vda — count 31, max_severity ~2.0e+09, low_data_flag=False
    - checkoutservice-0 / container.node-6.container_fs_reads./dev/vda — count 31, max_severity ~1.5e+09, low_data_flag=False
    - paymentservice-0 / container.node-6.container_fs_writes./dev/vda — count 31, max_severity ~1.5e+09, low_data_flag=False
    - paymentservice-1 / container.node-6.container_fs_writes./dev/vda — count 31, max_severity ~1.5e+09, low_data_flag=False
  - node-5 (examples)
    - adservice-2 / container.node-5.container_memory_failures.condition... — count 31, max_severity ~4.95e+10, low_data_flag=False
    - adservice-2 / container.node-5.container_memory_failures.hier... — count 31, max_severity ~4.95e+10, low_data_flag=False
    - cartservice2-0 / container.node-5.container_memory_failures.condition... — count 31, max_severity ~4.4e+10, low_data_flag=False
    - cartservice2-0 / container.node-5.container_memory_failures.hier... — count 31, max_severity ~4.4e+10, low_data_flag=False
    - frontend-1 / container.node-5.container_memory_failures.condition... — count 31, max_severity ~2.2e+10, low_data_flag=False

- Notes / interpretation:
  - Node-6 shows by far the largest number of anomalous metric points; many examples are filesystem read/write KPIs tied to /dev/vda across several services, suggesting heavy disk I/O anomalies on node-6.
  - Node-5 anomalies are dominated by container memory failure metrics (large severities) across a few services.
  - Example KPI pairs shown have low_data_flag=False (i.e., they had >=10 total data points in their full series), so thresholds were computed from sufficient historical data.
  - Outputs are limited to top nodes and a few example KPI pairs; full per-metric time series were not included. If you want, I can expand examples, list affected cmdb_ids per node, or show the underlying anomalous rows.

The original code execution output of IPython Kernel is also provided below for reference:

(  node_id  anomalous_point_count  distinct_cmdb_ids_affected  max_severity earliest_anomaly_timestamp_shanghai
0  node-6                  56801                          36  3.000000e+09            2022-03-21T16:30:00+0800
1  node-5                   9226                           6  4.950000e+10            2022-03-21T16:30:00+0800,   node_id            cmdb_id                                           kpi_name  anomaly_count  max_severity  low_data_flag
0  node-6   paymentservice-2      container.node-6.container_fs_writes./dev/vda             31  3.000000e+09          False
1  node-6   paymentservice-2       container.node-6.container_fs_reads./dev/vda             31  2.000000e+09          False
2  node-6  checkoutservice-0       container.node-6.container_fs_reads./dev/vda             31  1.500000e+09          False
3  node-6   paymentservice-0      container.node-6.container_fs_writes./dev/vda             31  1.500000e+09          False
4  node-6   paymentservice-1      container.node-6.container_fs_writes./dev/vda             31  1.500000e+09          False
5  node-5        adservice-2  container.node-5.container_memory_failures.con...             31  4.950000e+10          False
6  node-5        adservice-2  container.node-5.container_memory_failures.hie...             31  4.950000e+10          False
7  node-5     cartservice2-0  container.node-5.container_memory_failures.con...             31  4.400000e+10          False
8  node-5     cartservice2-0  container.node-5.container_memory_failures.hie...             31  4.400000e+10          False
9  node-5         frontend-1  container.node-5.container_memory_failures.con...             31  2.200000e+10          False)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse variables from prior runs if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware timestamp_shanghai
tz = pytz.timezone('Asia/Shanghai')
df_metric['timestamp_shanghai'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure global thresholds exist (calculated from full series)
try:
    thresholds
except NameError:
    grp_full = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    agg_counts = grp_full.count().rename('count')
    agg_p95 = grp_full.quantile(0.95).rename('p95')
    agg_p5 = grp_full.quantile(0.05).rename('p5')
    thresholds = pd.concat([agg_counts, agg_p95, agg_p5], axis=1).reset_index()
    thresholds['low_data_flag'] = thresholds['count'] < 10

# Incident window
start = tz.localize(pd.Timestamp('2022-03-21 16:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 17:00:00'))

mask_window = (df_metric['timestamp_shanghai'] >= start) & (df_metric['timestamp_shanghai'] <= end)
df_window = df_metric.loc[mask_window].copy()

# Focus on kpi_name containing 'node-6'
mask_node6 = df_window['kpi_name'].str.contains('node-6', na=False)
df_node6 = df_window.loc[mask_node6].copy()

# If no node-6 rows, return empty-safe results
if df_node6.empty:
    top_cmdbs_node6 = pd.DataFrame(columns=['cmdb_id','anomalous_point_count_on_node6','earliest_anomaly_timestamp_shanghai_for_that_cmdb','distinct_kpi_names_with_anomalies','low_data_pairs_count']).head(0)
    categories_node6 = pd.DataFrame(columns=['category','anomalous_point_count','example_kpi_name','example_low_data_flag','max_severity']).head(0)
else:
    # Merge global thresholds (count,p95,p5,low_data_flag) into node6 rows
    df_node6 = df_node6.merge(thresholds[['cmdb_id','kpi_name','count','p95','p5','low_data_flag']], on=['cmdb_id','kpi_name'], how='left')

    # Mark anomalies using global thresholds
    eps = 1e-9
    df_node6['anomaly_high'] = df_node6['p95'].notna() & (df_node6['value'] >= df_node6['p95'])
    df_node6['anomaly_low'] = df_node6['p5'].notna() & (df_node6['value'] <= df_node6['p5'])
    df_node6['anomaly'] = df_node6['anomaly_high'] | df_node6['anomaly_low']

    # severity
    df_node6['severity_high'] = np.where(df_node6['anomaly_high'], (df_node6['value'] - df_node6['p95']) / (df_node6['p95'] + eps), 0.0)
    df_node6['severity_low'] = np.where(df_node6['anomaly_low'], (df_node6['p5'] - df_node6['value']) / (df_node6['p5'] + eps), 0.0)
    df_node6['severity'] = df_node6[['severity_high','severity_low']].max(axis=1)

    # Keep only anomalous rows
    anoms_n6 = df_node6[df_node6['anomaly']].copy()

    if anoms_n6.empty:
        top_cmdbs_node6 = pd.DataFrame(columns=['cmdb_id','anomalous_point_count_on_node6','earliest_anomaly_timestamp_shanghai_for_that_cmdb','distinct_kpi_names_with_anomalies','low_data_pairs_count']).head(0)
        categories_node6 = pd.DataFrame(columns=['category','anomalous_point_count','example_kpi_name','example_low_data_flag','max_severity']).head(0)
    else:
        # 1) Per-cmdb aggregates on node-6
        per_cmdb = (anoms_n6.groupby('cmdb_id')
                    .agg(anomalous_point_count_on_node6=('anomaly','size'),
                         earliest_anomaly_timestamp=('timestamp_shanghai','min'),
                         distinct_kpi_names_with_anomalies=('kpi_name','nunique'),
                         low_data_pairs_count=('low_data_flag','sum'))
                    .reset_index())
        per_cmdb['earliest_anomaly_timestamp_shanghai_for_that_cmdb'] = per_cmdb['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
        per_cmdb = per_cmdb.drop(columns=['earliest_anomaly_timestamp'])
        top_cmdbs_node6 = per_cmdb.sort_values('anomalous_point_count_on_node6', ascending=False).head(20).reset_index(drop=True)

        # 2) Determine trace sources with anomalies
        trace_sources_set = set()
        if 'result_trace_anomalies' in globals() and isinstance(result_trace_anomalies, dict):
            df_ts = result_trace_anomalies.get('top_trace_sources_up_to_20')
            if df_ts is not None and not df_ts.empty:
                trace_sources_set.update(df_ts['source'].astype(str).tolist())
            # also include any short list if present
            df_ts2 = result_trace_anomalies.get('top_trace_sources_list_up_to_10')
            if df_ts2 is not None and not df_ts2.empty:
                trace_sources_set.update(df_ts2['source'].astype(str).tolist())

        # If trace_sources_set is empty, attempt light re-evaluation of trace.csv to find any trace sources with anomalies
        if len(trace_sources_set) == 0:
            try:
                df_trace
            except NameError:
                df_trace = pd.read_csv('trace.csv')
                df_trace['timestamp_shanghai'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
            # quick global thresholds per trace_name
            grp = df_trace.groupby('trace_name')['value']
            thr_counts = grp.count().rename('count')
            thr_p95 = grp.quantile(0.95).rename('p95')
            thr_p5 = grp.quantile(0.05).rename('p5')
            thr = pd.concat([thr_counts, thr_p95, thr_p5], axis=1).reset_index()
            # mark anomalies same as earlier: error_rate & duration -> high; row_count -> low
            df_tr = df_trace.merge(thr, on='trace_name', how='left')
            is_error_rate = df_tr['trace_name'].str.contains('error_rate', na=False)
            is_row_count = df_tr['trace_name'].str.contains('row_count', na=False)
            is_duration = df_tr['trace_name'].str.contains('duration_mean|duration_p95', regex=True, na=False)
            df_tr['anomaly_high'] = ((is_error_rate | is_duration) & (df_tr['value'] >= df_tr['p95']))
            df_tr['anomaly_low'] = (is_row_count & (df_tr['value'] <= df_tr['p5']))
            df_tr['anomaly'] = df_tr['anomaly_high'] | df_tr['anomaly_low']
            # extract source
            df_tr['source'] = df_tr['trace_name'].str.extract(r'^trace\.from_([^\.]+)\.', expand=False)
            trace_sources_set.update(df_tr.loc[df_tr['anomaly'], 'source'].dropna().unique().tolist())

        # Add boolean column to top_cmdbs_node6 indicating trace source anomalies
        top_cmdbs_node6['trace_source_has_anomalies'] = top_cmdbs_node6['cmdb_id'].astype(str).apply(lambda x: x in trace_sources_set)

        # 3) KPI-category aggregation for node-6 anomalous KPI names
        def categorize_kpi(name):
            n = (name or '').lower()
            if '/dev' in n or 'fs' in n:
                return 'fs_io'
            if 'memory' in n:
                return 'memory'
            if 'cpu' in n:
                return 'cpu'
            if ('network' in n) or ('transmit' in n) or ('receive' in n):
                return 'network'
            if 'sockets' in n:
                return 'sockets'
            return 'other'

        anoms_n6['category'] = anoms_n6['kpi_name'].apply(categorize_kpi)

        per_cat = (anoms_n6.groupby('category')
                   .agg(anomalous_point_count=('anomaly','size'),
                        example_kpi_name=('kpi_name', 'first'),
                        max_severity=('severity','max'))
                   .reset_index())

        # compute example_low_data_flag for example_kpi_name per category
        def example_low_flag(row):
            ex = row['example_kpi_name']
            return bool(anoms_n6.loc[anoms_n6['kpi_name'] == ex, 'low_data_flag'].any())

        per_cat['example_low_data_flag'] = per_cat.apply(example_low_flag, axis=1)
        categories_node6 = per_cat.sort_values('anomalous_point_count', ascending=False).head(10).reset_index(drop=True)
        categories_node6 = categories_node6[['category','anomalous_point_count','example_kpi_name','example_low_data_flag','max_severity']]

# Display compact results: top_cmdbs_node6 (up to 20) and categories_node6 (up to 10)
top_cmdbs_node6, categories_node6

```
Out[6]:
```
```python
# Build a concise plain-English summary from the node-6 analysis results
top = top_cmdbs_node6  # reused from prior execution
cats = categories_node6

# Prepare lists for summary
top_list = []
for i, r in top.head(10).iterrows():
    cm = r['cmdb_id']
    cnt = int(r['anomalous_point_count_on_node6'])
    distinct_kpis = int(r['distinct_kpi_names_with_anomalies'])
    earliest = r['earliest_anomaly_timestamp_shanghai_for_that_cmdb']
    trace_flag = bool(r['trace_source_has_anomalies'])
    low_pairs = int(r.get('low_data_pairs_count', 0))
    top_list.append(f"{i+1}. {cm}: {cnt} anomalies, {distinct_kpis} KPI names affected, earliest {earliest}, trace_source_has_anomalies={trace_flag}, low_data_pairs={low_pairs}")

# Category summaries (top up to 10)
cat_list = []
for i, r in cats.head(10).iterrows():
    cat = r['category']
    cnt = int(r['anomalous_point_count'])
    ex = r['example_kpi_name']
    maxs = float(r['max_severity'])
    ldf = bool(r['example_low_data_flag'])
    cat_list.append(f"{i+1}. {cat}: {cnt} anomalies, example='{ex}', max_severity={maxs:.4g}, example_low_data={ldf}")

summary_text = (
    "Node-6 localized anomalies (incident window 2022-03-21 16:30–17:00 Asia/Shanghai):\n\n"
    "Top affected cmdb_id on node-6 (showing top 10 of up to 20):\n" +
    "\n".join(top_list) +
    "\n\nKPI category breakdown on node-6 (top categories):\n" +
    "\n".join(cat_list) +
    "\n\nKey takeaways:\n"
    "- Many services (pods) show large and similar anomalous counts on node-6 (each top cmdb_id ~1.6k–1.7k anomalous points).\n"
    "- All top cmdb_ids have earliest anomalies at 2022-03-21T16:30:00+08:00 and low_data_pairs_count=0 (thresholds based on sufficient data).\n"
    "- Trace-side anomalies overlap for several services (trace_source_has_anomalies=True for some recommendation, frontend, and checkout entries), indicating correlated trace and metric issues for those components.\n"
    "- Metric categories most impacted on node-6 are: fs_io (largest), memory, other (file-descriptor related), network, cpu, then sockets — suggesting I/O and memory-related problems on node-6.\n"
    "- If you want, I can expand the full top-20 cmdb list, show the anomalous KPI names per specific cmdb_id, or export the anomalous rows for deeper inspection."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  anomalous_point_count_on_node6  distinct_kpi_names_with_anomalies  low_data_pairs_count earliest_anomaly_timestamp_shanghai_for_that_cmdb  trace_source_has_anomalies
0    productcatalogservice-0                            1736                                 64                     0                          2022-03-21T16:30:00+0800                       False
1             emailservice-0                            1727                                 64                     0                          2022-03-21T16:30:00+0800                       False
2             emailservice-1                            1725                                 64                     0                          2022-03-21T16:30:00+0800                       False
3            emailservice2-0                            1724                                 64                     0                          2022-03-21T16:30:00+0800                       False
4    recommendationservice-0                            1724                                 64                     0                          2022-03-21T16:30:00+0800                        True
5             emailservice-2                            1723                                 64                     0                          2022-03-21T16:30:00+0800                       False
6   recommendationservice2-0                            1722                                 64                     0                          2022-03-21T16:30:00+0800                        True
7          shippingservice-0                            1721                                 64                     0                          2022-03-21T16:30:00+0800                       False
8         shippingservice2-0                            1721                                 64                     0                          2022-03-21T16:30:00+0800                       False
9    recommendationservice-2                            1718                                 64                     0                          2022-03-21T16:30:00+0800                        True
10                frontend-0                            1714                                 64                     0                          2022-03-21T16:30:00+0800                        True
11          paymentservice-0                            1711                                 64                     0                          2022-03-21T16:30:00+0800                       False
12         shippingservice-1                            1709                                 63                     0                          2022-03-21T16:30:00+0800                       False
13   recommendationservice-1                            1709                                 64                     0                          2022-03-21T16:30:00+0800                        True
14   productcatalogservice-2                            1708                                 63                     0                          2022-03-21T16:30:00+0800                       False
15          paymentservice-1                            1707                                 64                     0                          2022-03-21T16:30:00+0800                       False
16   productcatalogservice-1                            1688                                 64                     0                          2022-03-21T16:30:00+0800                       False
17        checkoutservice2-0                            1674                                 64                     0                          2022-03-21T16:30:00+0800                        True
18         checkoutservice-1                            1671                                 64                     0                          2022-03-21T16:30:00+0800                        True
19               adservice-1                            1646                                 64                     0                          2022-03-21T16:30:00+0800                       False,   category  anomalous_point_count                                   example_kpi_name  example_low_data_flag  max_severity
0    fs_io                  21266  container.node-6.container_cpu_cfs_throttled_p...                  False  3.000000e+09
1   memory                  13979            container.node-6.container_memory_cache                  False  1.338000e+02
2    other                  10591        container.node-6.container_file_descriptors                  False  1.285714e+00
3  network                   5084  container.node-6.container_network_receive_err...                  False  2.479965e+01
4      cpu                   4837    container.node-6.container_cpu_load_average_10s                  False  1.013455e+02
5  sockets                   1044                 container.node-6.container_sockets                  False  1.666667e-01)```
```